# 第6讲：时间序列分析与交通状态预测（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：4分钟

        ## 目标

        1. 把WINDOW从3改为6，比较移动平均的平滑程度和误差；
2. 把ALPHA从0.4改为0.8，说明指数平滑对新变化的响应怎样改变；
3. 找出AR(4)绝对误差最大的三个时段，并结合原序列解释可能原因；

        学生版的代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。教师版和参考答案版已保存一次完整运行结果。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 读取D03速度序列并按时间切分

In [ ]:
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"])
series = (traffic.loc[traffic["detector_id"] == "D03", ["timestamp", "speed_kmh"]]
          .sort_values("timestamp").set_index("timestamp"))
WINDOW = 3       # TODO：改为6，比较窗口变化
ALPHA = 0.4      # TODO：改为0.8，比较响应速度
AR_LAGS = 4
CUT_TIME = pd.Timestamp("2026-03-12 00:00")
train = series.loc[series.index < CUT_TIME, "speed_kmh"]
test = series.loc[series.index >= CUT_TIME, "speed_kmh"]
print({"记录数": len(series), "时间间隔": pd.infer_freq(series.index),
       "训练期": f"{train.index.min()} 至 {train.index.max()}",
       "检验期": f"{test.index.min()} 至 {test.index.max()}"})
display(series.loc["2026-03-04 06:45":"2026-03-04 07:45"])

## 3. 计算移动平均和一次指数平滑

In [ ]:
y = series["speed_kmh"]
forecasts = pd.DataFrame({"actual": y})
forecasts["latest"] = y.shift(1)
forecasts["sma"] = y.shift(1).rolling(WINDOW).mean()

weights = (np.array([0.2, 0.3, 0.5]) if WINDOW == 3
           else np.arange(1, WINDOW + 1, dtype=float))
weights = weights / weights.sum()
forecasts["wma"] = y.shift(1).rolling(WINDOW).apply(
    lambda values: np.dot(values, weights), raw=True)

ses = pd.Series(np.nan, index=y.index, dtype=float)
level = float(y.iloc[0])
for i in range(1, len(y)):
    ses.iloc[i] = level
    level = ALPHA * float(y.iloc[i]) + (1 - ALPHA) * level
forecasts["ses"] = ses
print("WMA权重（由远到近）：", np.round(weights, 3))
display(forecasts.loc["2026-03-04 06:45":"2026-03-04 07:45"].round(2))

## 4. 拟合AR(4)并在同一检验期比较

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error

ar_fit = AutoReg(train, lags=AR_LAGS, trend="c", old_names=False).fit()
ar_pred = pd.Series(np.nan, index=y.index, dtype=float)
params = np.asarray(ar_fit.params, dtype=float)
for timestamp in test.index:
    pos = y.index.get_loc(timestamp)
    lag_values = y.iloc[pos-AR_LAGS:pos].to_numpy()[::-1]
    ar_pred.loc[timestamp] = params[0] + np.dot(params[1:], lag_values)
forecasts["ar4"] = ar_pred

comparison = forecasts.loc[test.index].dropna().copy()
rows = []
for method in ["latest", "sma", "wma", "ses", "ar4"]:
    rows.append({"method": method,
                 "MAE": mean_absolute_error(comparison["actual"], comparison[method]),
                 "RMSE": mean_squared_error(comparison["actual"], comparison[method]) ** 0.5})
scores = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
display(scores.round(3))
print("AR(4)参数：", dict(zip(["const", "L1", "L2", "L3", "L4"], np.round(params, 3))))
scores.to_csv(OUTPUT_DIR / "lesson06_method_comparison.csv", index=False)

## 5. 检查最大误差时段并自检

In [ ]:
comparison["ar4_abs_error"] = (comparison["actual"] - comparison["ar4"]).abs()
largest_errors = comparison.nlargest(3, "ar4_abs_error")[
    ["actual", "ar4", "ar4_abs_error"]]
display(largest_errors.round(2))
largest_errors.to_csv(OUTPUT_DIR / "lesson06_largest_errors.csv")

plot_data = comparison.iloc[:96]
ax = plot_data[["actual", "latest", "sma", "ses", "ar4"]].plot(
    figsize=(10, 4.8), linewidth=1.5, title="D03: one-step speed forecasts on the same test period")
ax.set_ylabel("Speed (km/h)"); ax.set_xlabel("Time")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson06_forecast_comparison.png"); plt.show()

checks = {
    "训练早于检验": train.index.max() < test.index.min(),
    "五种方法使用同一检验期": len(scores) == 5 and len(comparison) == len(test),
    "预测没有缺失": comparison[["latest", "sma", "wma", "ses", "ar4"]].notna().all().all(),
    "三个输出文件已生成": all((OUTPUT_DIR / name).exists() for name in [
        "lesson06_method_comparison.csv", "lesson06_largest_errors.csv",
        "lesson06_forecast_comparison.png"]),
}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 把WINDOW从3改为6，比较移动平均的平滑程度和误差
- 把ALPHA从0.4改为0.8，说明指数平滑对新变化的响应怎样改变
- 找出AR(4)绝对误差最大的三个时段，并结合原序列解释可能原因

        **预期结果：** 五种方法的误差比较表、同一检验期预测曲线和三个最大误差时段。

        **完成标准：** 所有方法使用同一时间检验期；能解释窗口和α怎样分配历史权重；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。